# Ekstraksi Sentinel-2 L2A
**Joni Kumar Meghwar · 240411100234**

Adapted from the supplied Muhammad Ainul Fuady/PSD UTS tiled extraction. This notebook uses your cleaned digitization, not the reference samples. Run the plan before submitting remote jobs.

In [ ]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uts/code/pipeline.py").exists()), None)
if root is None: raise RuntimeError("Open this notebook inside the extracted project folder")
sys.path.insert(0, str(root / "uts/code"))
from pipeline import *

## 1. Inspect your samples
Original features 70, 71 and 148 are excluded pending redigitization; nine geometries repaired. B8A is included alongside B08. Shared pixels between overlapping polygons are removed.

In [ ]:
display(pd.read_csv(DEPLOY / "sample_summary.csv"))
display(pd.read_csv(DEPLOY / "geometry_issues.csv"))

## 2. Plan tiles without login
No remote processing jobs are submitted here.

In [ ]:
import subprocess
subprocess.run([sys.executable, str(root / "uts/code/crawling.py"), "--plan"], check=True)

## 3. Download and extract actual satellite data
The next cell requests Copernicus login through a device link and submits real jobs. Read your account processing credits first. Dates default to September 2026. Keep this running until completion. Outputs are cached for resumable jobs; changing settings requires a fresh output folder.

In [ ]:
subprocess.run([sys.executable, str(root / "uts/code/crawling.py")], check=True)

## 4. Inspect extraction
Every CSV row is a real valid Sentinel-2 pixel tied to its source polygon. Review missing polygons, clouds and sample sizes before modeling.

In [ ]:
pixels = pd.read_csv(UTS / "data/csv/hasil_s2_piksel/piksel_s2.csv")
display(pixels.head())
display(pixels.groupby("label_teks").agg(pixels=("poligon_id","size"), polygons=("poligon_id","nunique")))